In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path
from aquarel import load_theme

theme = load_theme("minimal_light")
theme.apply()
theme.apply_transforms()
import matplotlib as mpl
mpl.rcParams['axes.spines.left'] = True
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.bottom'] = True
mpl.rcParams['axes.edgecolor'] = 'grey'

In [ ]:
FIGURE_DATA = Path("../figure_data")
df = pd.read_csv(FIGURE_DATA / 'simphyni_effects.csv')

In [ ]:
### Figure S7B

es_pos_df = df[df['label'] == 1]
es_neg_df = df[df['label'] == -1]
unique_powers = sorted(df['es_input'].unique())

mean_es_pos = [es_pos_df[es_pos_df['es_input'] == p]['inferred_effect_size'].mean() for p in unique_powers]
std_es_pos  = [es_pos_df[es_pos_df['es_input'] == p]['inferred_effect_size'].std()  for p in unique_powers]

mean_es_neg = [es_neg_df[es_neg_df['es_input'] == p]['inferred_effect_size'].mean() for p in unique_powers]
std_es_neg  = [es_neg_df[es_neg_df['es_input'] == p]['inferred_effect_size'].std()  for p in unique_powers]

# Target y-values for intersection (E. coli accessory genome effect sizes)
y_target_p_u = 5.71
y_target_p_l = 1.95
y_target_n_u = 3.71
y_target_n_l = 1.95

power_intersect_pos = 0.75
power_intersect_neg = 0.75

unique_powers = np.array(unique_powers)

# --- Create subplots for Positive and Negative Associations ---
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=False)
ticks_pos = np.sort(np.unique(np.concatenate([unique_powers, [power_intersect_pos]])))
ticks_neg = np.sort(np.unique(np.concatenate([unique_powers, [power_intersect_neg]])))

axes[0].errorbar(unique_powers, mean_es_pos, yerr=std_es_pos,
                 fmt="-o", capsize=4, markersize=5, color="#ff711f",
                 label="Positive Associations")

axes[0].axhline(y_target_p_u, color="gray", linestyle="dashed", alpha=0.5, linewidth=2)
axes[0].axhline(y_target_p_l, color="gray", linestyle="dashed", alpha=0.5, linewidth=2)
axes[0].set_xticks(ticks_pos)
axes[0].set_xlabel("Interaction Strength")
axes[0].set_ylabel("Effect Size")
axes[0].set_title("Positive Associations")
axes[0].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

# Plot Negative Associations
axes[1].errorbar(unique_powers, mean_es_neg, yerr=std_es_neg,
                 fmt="-o", capsize=4, markersize=5, color="#0a6bf2",
                 label="Negative Associations")

axes[1].axhline(y_target_n_u, color="gray", linestyle="dashed", alpha=0.5, linewidth=2)
axes[1].axhline(y_target_n_l, color="gray", linestyle="dashed", alpha=0.5, linewidth=2)
axes[1].set_xticks(ticks_neg)
axes[1].set_xlabel("Interaction Strength")
axes[1].set_title("Negative Associations")

for ax, highlight_x, highlight_color in zip(axes, [power_intersect_pos, power_intersect_neg],
                                             ["#ff711f", "#0a6bf2"]):
    xticks = ax.get_xticks()
    fmt_labels = []
    for t in xticks:
        if abs(t - round(t)) < 1e-8:
            lbl = f"{int(round(t))}"
        else:
            lbl = f"{t:.2f}".rstrip('0').rstrip('.')
        fmt_labels.append(lbl)
    ax.set_xticklabels(fmt_labels, rotation=45, ha='right')
    closest_idx = np.argmin(np.abs(xticks - highlight_x))
    if abs(xticks[closest_idx] - highlight_x) < 1e-6:
        txt = ax.get_xticklabels()[closest_idx]
        txt.set_fontweight('bold')
        txt.set_color(highlight_color)
    else:
        ax.annotate(f"{highlight_x:.2f}", xy=(highlight_x, ax.get_ylim()[0]),
                    xycoords=('data', 'data'),
                    xytext=(0, -20), textcoords='offset points',
                    ha='center', va='top', color=highlight_color, fontsize=9)

plt.tight_layout()
plt.savefig('fig.svg', format='svg')
plt.show()

# --- Separate Legend Plot ---

fig_legend, ax_leg = plt.subplots(figsize=(3, 3))
ax_leg.axis('off')
horizontal_line = mlines.Line2D([], [], color='gray', linestyle='dashed', alpha=0.5, linewidth=2)
errorbar_pos = mlines.Line2D([], [], color='#ff711f', marker='o', linestyle='-', markersize=5)
errorbar_neg = mlines.Line2D([], [], color='#0a6bf2', marker='o', linestyle='-', markersize=5)
legend_lines = [
    errorbar_pos,
    errorbar_neg,
    horizontal_line,
]
legend_labels = [
    "Positive Associations\n(Mean \u00b1 SD)",
    "Negative Associations\n(Mean \u00b1 SD)",
    "5th and 95th Percentile Effect Size\n(E. coli Accessory Genome)",
]
ax_leg.legend(legend_lines, legend_labels, loc='center', frameon=False)
plt.tight_layout()
plt.savefig('figleg.svg', format='svg')
plt.show()